# S6E11 GPU · TabM-mini trên CUDA

Thử nghiệm trên dữ liệu chính thức của cuộc thi. Cách chia 5 fold và seed 42 giống đợt trước để so sánh trực tiếp.

Mở notebook từ bản sao repo và chọn kernel Python có thư viện trong
[Hướng dẫn chuẩn bị](README.md). Dữ liệu chính thức được đọc cục bộ và đối chiếu SHA-256.

## 1. Dữ liệu và môi trường GPU

Nguồn nhãn là `train.csv` chính thức của S6E11. Mã kiểm tra SHA-256 phải khớp
bản dữ liệu của repo. Tập kiểm tra của Kaggle không có nhãn và ROC AUC trong
notebook được đo trên các giao dịch giữ lại từ tập huấn luyện có nhãn.
`id` chỉ dùng để truy vết. Mỗi mô hình xác nhận thiết bị GPU trước khi fit.

In [ ]:
import hashlib
import json
import os
from pathlib import Path
from subprocess import check_output
import sys
from time import perf_counter
from zipfile import ZipFile

import numpy as np
import pandas as pd
import sklearn
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold, train_test_split

SEED = 42
SMOKE = os.environ.get('S6E11_SMOKE') == '1'

def locate_repo():
    override = os.environ.get('S6E11_REPO')
    start = Path.cwd().resolve()
    roots = (Path(override).expanduser().resolve(),) if override else (start, *start.parents)
    for root in roots:
        if (root / 'configs' / 'default.json').is_file() and (root / 'data' / 'manifest.json').is_file():
            return root
    raise FileNotFoundError('Mở notebook trong repo S6E11 hoặc đặt S6E11_REPO.')


def locate_data(repo):
    override = os.environ.get('S6E11_DATA_DIR')
    roots = (Path(override).expanduser().resolve(),) if override else (repo / 'data', repo.parent)
    for root in roots:
        if any((root / name).is_file() for name in ('train.csv', 'playground-series-s6e11.zip')):
            return root
    raise FileNotFoundError('Cần train.csv hoặc ZIP S6E11 chính thức. Đặt S6E11_DATA_DIR.')


REPO = locate_repo()
DATA_DIR = locate_data(REPO)
default_output = REPO / 's6e11_gpu' / 'model_notebooks' / 'local_results'
OUTPUT_ROOT = Path(os.environ.get('S6E11_NOTEBOOK_OUTPUT', str(default_output))).expanduser().resolve()
ROUND1_DIR = OUTPUT_ROOT / 'round1'
ROUND2_DIR = OUTPUT_ROOT / 'round2'
CONFIG = json.loads((REPO / 'configs' / 'default.json').read_text(encoding='utf-8-sig'))
MANIFEST = json.loads((REPO / 'data' / 'manifest.json').read_text(encoding='utf-8-sig'))
TARGET = CONFIG['target']
NUMERIC = CONFIG['numeric_columns']
FLAGS = CONFIG['flag_columns']
CATEGORICAL = CONFIG['categorical_columns']

GPU_NAME = check_output(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
                        text=True, timeout=15).splitlines()[0].strip()
if 'RTX 4060' not in GPU_NAME:
    raise RuntimeError(f'GPU hiện tại khác RTX 4060 đã kiểm tra: {GPU_NAME}')
print(f'GPU phát hiện: {GPU_NAME}')

def load_competition_train():
    expected_hash = MANIFEST['input_files']['train.csv']['sha256']
    csv_path = DATA_DIR / 'train.csv'
    if csv_path.is_file():
        with csv_path.open('rb') as stream:
            digest = hashlib.file_digest(stream, 'sha256').hexdigest()
        if digest != expected_hash:
            raise ValueError('SHA-256 của train.csv không khớp manifest')
        return pd.read_csv(csv_path), digest

    with ZipFile(DATA_DIR / 'playground-series-s6e11.zip') as archive:
        with archive.open('train.csv') as stream:
            digest = hashlib.file_digest(stream, 'sha256').hexdigest()
        if digest != expected_hash:
            raise ValueError('SHA-256 của train.csv trong ZIP không khớp manifest')
        with archive.open('train.csv') as stream:
            return pd.read_csv(stream), digest

frame, DATA_HASH = load_competition_train()
feature_names = NUMERIC + FLAGS + CATEGORICAL
required = ['id', TARGET, *feature_names]
if (set(frame.columns) != set(required) or len(frame) != MANIFEST['train_rows']
        or frame.id.duplicated().any() or frame[required].isna().any().any()
        or set(frame[TARGET].unique()) != {0, 1}):
    raise ValueError('Dữ liệu huấn luyện không khớp schema đã kiểm tra của repo')
if SMOKE:
    _, frame = train_test_split(
        frame, test_size=20_000, stratify=frame[TARGET], random_state=SEED,
    )
    frame = frame.reset_index(drop=True)
y = frame[TARGET].to_numpy(dtype=np.uint8)
ids = frame['id'].to_numpy(dtype=np.int64)
X = frame[feature_names]
raw_numeric = X[NUMERIC + FLAGS].to_numpy(dtype=np.float32)
if not np.isfinite(raw_numeric).all():
    raise ValueError('Đặc trưng số có giá trị không hữu hạn')
print(f'Số dòng: {len(y):,} | nhãn gian lận: {y.sum():,} ({y.mean():.2%})')

## 2. Năm phần chia phân tầng

Dùng `StratifiedKFold` với seed 42 như repo để giữ tỷ lệ gian lận giữa các phần.
Mỗi giao dịch được chấm một lần bởi mô hình học từ bốn phần còn lại.
Số vòng hoặc số epoch được cố định trước khi đánh giá phần giữ lại.

In [ ]:
n_folds = 2 if SMOKE else 5
splitter = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=SEED)
splits = list(splitter.split(X, y))
assignment = np.full(len(y), -1, dtype=np.int8)
for fold, (_, val_idx) in enumerate(splits):
    assignment[val_idx] = fold
if (assignment < 0).any():
    raise RuntimeError('Có giao dịch chưa được gán fold')
SPLIT_HASH = hashlib.sha256(ids.astype('<i8').tobytes() + assignment.tobytes()).hexdigest()
print(f'{n_folds} fold | mã kiểm tra cách chia: {SPLIT_HASH[:16]}')

## 3. Chuẩn hóa đầu vào cho mạng nơ-ron

Trung bình, độ lệch chuẩn và bảng giá trị phân loại được xác định từ phần
huấn luyện. Các giá trị phân loại chưa gặp dùng mã 0. Mạng học các vectơ biểu
diễn (embeddings) của cột phân loại trực tiếp từ nhãn huấn luyện.

In [ ]:
def prepare_dl_fold(train_idx, val_idx):
    raw_train, raw_val = raw_numeric[train_idx], raw_numeric[val_idx]
    center = raw_train.mean(axis=0, dtype=np.float64).astype(np.float32)
    scale = raw_train.std(axis=0, dtype=np.float64).astype(np.float32)
    scale[scale < 1e-7] = 1.0
    num_train = (raw_train - center) / scale
    num_val = (raw_val - center) / scale
    cat_train = np.empty((len(train_idx), len(CATEGORICAL)), dtype=np.int64)
    cat_val = np.empty((len(val_idx), len(CATEGORICAL)), dtype=np.int64)
    sizes = []
    for col, name in enumerate(CATEGORICAL):
        values = X[name].to_numpy()
        vocab = pd.Index(pd.unique(values[train_idx]))
        sizes.append(len(vocab) + 1)
        cat_train[:, col] = vocab.get_indexer(values[train_idx]) + 1
        cat_val[:, col] = vocab.get_indexer(values[val_idx]) + 1
    return (num_train, cat_train), (num_val, cat_val), sizes

## TabM-mini trên GPU

TabM-mini là một tập hợp (ensemble) gồm 8 mạng con dùng chung phần lớn
trọng số. Mô hình nhận biến số đã chuẩn hóa và dùng mã hóa one-hot cho
biến phân loại. Từ điển phân loại được lập từ phần huấn luyện của mỗi
fold. Mã 0 dành cho giá trị chưa gặp.

Notebook dùng [gói TabM chính thức](https://github.com/yandex-research/tabm)
và khởi tạo trọng số mới ở mỗi fold. Cấu hình cố định gồm hai khối
128 đơn vị, dropout 0,1 và AdamW.

In [ ]:
from importlib.metadata import version
from time import perf_counter

import torch
from torch import nn

from tabm import TabM


TABM_EPOCHS = 1 if SMOKE else 5
TABM_BATCH_SIZE = 1024
TABM_MODEL_SETTINGS = {
    "arch_type": "tabm-mini",
    "n_blocks": 2,
    "d_block": 128,
    "dropout": 0.1,
    "k": 8,
}
TABM_OPTIMIZER_SETTINGS = {"lr": 0.002, "weight_decay": 0.0003, "fused": True}

## Huấn luyện và dự đoán trong một fold

Mỗi mạng con có một điểm số (logit) cho mỗi giao dịch. Hàm mất mát
binary cross-entropy dùng logit của từng mạng con rồi lấy trung bình.
Khi dự đoán, hàm sigmoid chuyển từng logit thành xác suất trước khi
lấy trung bình của 8 xác suất.

Mỗi fold huấn luyện trong 5 epoch (5 lượt qua phần huấn luyện), hoặc
1 epoch khi chạy thử. Phần giữ lại chỉ dùng để dự đoán sau khi
huấn luyện. Thời gian fit đo từ lúc khởi tạo mô hình tới khi hoàn tất
cập nhật trọng số trên CUDA. GPU hỗ trợ bfloat16 sẽ dùng độ chính xác
hỗn hợp tự động (AMP).

In [ ]:
def fit_predict(train_idx, val_idx, fold):
    if not torch.cuda.is_available():
        raise RuntimeError("TabM-mini cần PyTorch có CUDA và GPU NVIDIA đang hoạt động.")
    device = torch.device("cuda", torch.cuda.current_device())
    gpu_name = torch.cuda.get_device_name(device)
    if gpu_name != GPU_NAME:
        raise RuntimeError(f"Thiết bị PyTorch không khớp GPU đã kiểm tra: {gpu_name}")

    seed = SEED + int(fold)
    torch.manual_seed(seed)
    generator = torch.Generator(device=device).manual_seed(seed)
    use_bfloat16 = torch.cuda.is_bf16_supported()

    (train_num, train_cat), (val_num, val_cat), category_sizes = prepare_dl_fold(
        train_idx, val_idx
    )
    train_num = torch.as_tensor(train_num, dtype=torch.float32, device=device)
    train_cat = torch.as_tensor(train_cat, dtype=torch.long, device=device)
    train_y = torch.as_tensor(y[train_idx], dtype=torch.float32, device=device)

    torch.cuda.synchronize(device)
    start = perf_counter()
    model = TabM.make(
        n_num_features=train_num.shape[1],
        cat_cardinalities=list(map(int, category_sizes)),
        d_out=1,
        **TABM_MODEL_SETTINGS,
    ).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), **TABM_OPTIMIZER_SETTINGS)
    criterion = nn.BCEWithLogitsLoss()

    model.train()
    for _ in range(TABM_EPOCHS):
        order = torch.randperm(len(train_y), device=device, generator=generator)
        for rows in order.split(TABM_BATCH_SIZE):
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast("cuda", dtype=torch.bfloat16, enabled=use_bfloat16):
                logits = model(train_num[rows], train_cat[rows]).squeeze(-1)
                targets = train_y[rows, None].expand_as(logits)
                loss = criterion(logits, targets)
            loss.backward()
            optimizer.step()

    torch.cuda.synchronize(device)
    fit_seconds = perf_counter() - start

    val_num = torch.as_tensor(val_num, dtype=torch.float32, device=device)
    val_cat = torch.as_tensor(val_cat, dtype=torch.long, device=device)
    scores = torch.empty(len(val_idx), dtype=torch.float32, device=device)
    model.eval()
    with torch.inference_mode():
        for start in range(0, len(val_idx), TABM_BATCH_SIZE):
            rows = slice(start, start + TABM_BATCH_SIZE)
            with torch.autocast("cuda", dtype=torch.bfloat16, enabled=use_bfloat16):
                logits = model(val_num[rows], val_cat[rows]).squeeze(-1)
            scores[rows] = logits.float().sigmoid().mean(dim=1)

    return scores.cpu().numpy(), {
        "device": f"{device}: {gpu_name} (PyTorch CUDA)",
        "fit_seconds": float(fit_seconds),
        "settings": {
            **TABM_MODEL_SETTINGS,
            "epochs": TABM_EPOCHS,
            "batch_size": TABM_BATCH_SIZE,
            "optimizer": "AdamW",
            **TABM_OPTIMIZER_SETTINGS,
            "precision": "bfloat16 AMP" if use_bfloat16 else "float32",
            "tabm_version": version("tabm"),
        },
    }

## Đánh giá

ROC AUC đo chất lượng xếp hạng giao dịch gian lận. Điểm OOF gộp dùng các dự
đoán từ cả năm phần giữ lại. Average Precision (AP) tóm tắt độ chính xác của
cảnh báo khi thay đổi ngưỡng. Tỷ lệ gian lận là mốc tham khảo cho AP ngẫu nhiên.
Thời gian fit được đo trong từng mô hình. Thời gian toàn lượt còn gồm tiền xử
lý và suy luận.

In [ ]:
MODEL_SLUG = 'tabm-mini'
output_dir = ROUND2_DIR / ('smoke' if SMOKE else 'full')
output_dir.mkdir(parents=True, exist_ok=True)
oof = np.full(len(y), np.nan, dtype=np.float64)
fold_metrics = []
run_start = perf_counter()
for fold, (train_idx, val_idx) in enumerate(splits[:1] if SMOKE else splits):
    print(f'Bắt đầu fold {fold + 1}/{1 if SMOKE else len(splits)}', flush=True)
    if np.isfinite(oof[val_idx]).any():
        raise RuntimeError('Fold xác thực bị trùng')
    scores, details = fit_predict(train_idx, val_idx, fold)
    scores = np.asarray(scores, dtype=np.float64)
    if (scores.shape != (len(val_idx),) or not np.isfinite(scores).all()
            or (scores < 0).any() or (scores > 1).any()):
        raise RuntimeError(f'Fold {fold + 1} trả về xác suất không hợp lệ')
    if 'RTX 4060' not in details['device']:
        raise RuntimeError(f"Fold {fold + 1} không xác minh được RTX 4060: {details['device']}")
    oof[val_idx] = scores
    fold_auc = float(roc_auc_score(y[val_idx], scores))
    record = dict(details, fold=fold + 1, validation_rows=len(val_idx), roc_auc=fold_auc)
    fold_metrics.append(record)
    print(f"Fold {fold + 1}: ROC AUC {fold_auc:.6f} | {details['fit_seconds']:.2f}s | {details['device']}", flush=True)

evaluated = np.isfinite(oof)
if not SMOKE and not evaluated.all():
    raise RuntimeError('Dự đoán OOF chưa phủ toàn bộ tập huấn luyện')
metrics = {
    'model': MODEL_SLUG,
    'status': 'completed',
    'smoke': SMOKE,
    'models_use_llm': False,
    'rows': int(evaluated.sum()),
    'positives': int(y[evaluated].sum()),
    'train_sha256': DATA_HASH,
    'split_sha256': SPLIT_HASH,
    'roc_auc': float(roc_auc_score(y[evaluated], oof[evaluated])),
    'average_precision': float(average_precision_score(y[evaluated], oof[evaluated])),
    'fit_seconds': float(sum(record['fit_seconds'] for record in fold_metrics)),
    'wall_seconds': float(perf_counter() - run_start),
    'folds': fold_metrics,
    'versions': {name: sys.modules[name].__version__
                 for name in ('numpy', 'pandas', 'sklearn', 'torch', 'lightgbm', 'xgboost')
                 if name in sys.modules},
}
result = output_dir / f'{MODEL_SLUG}.json'
temp = result.with_suffix('.tmp')
temp.write_text(json.dumps(metrics, indent=2, ensure_ascii=False) + '\n', encoding='utf-8')
temp.replace(result)
print(f"{'Smoke AUC' if SMOKE else 'ROC AUC OOF'}: {metrics['roc_auc']:.6f}")
print(f"AP: {metrics['average_precision']:.6f} | Fit: {metrics['fit_seconds']:.2f}s")

oof_path = output_dir / f'{MODEL_SLUG}-oof.npz'
temp_oof = oof_path.with_suffix('.tmp')
with temp_oof.open('wb') as stream:
    np.savez(stream, ids=ids[evaluated], fold=assignment[evaluated],
             prediction=oof[evaluated], train_sha256=DATA_HASH, split_sha256=SPLIT_HASH)
temp_oof.replace(oof_path)
print(f'Đã lưu dự đoán theo giao dịch: {oof_path.name}')